# Classification with SNEPPX-Algo

Build a small MLP classifier on synthetic 8x8 data using `SneppX_ALG`'s `nn`
module and `AdamW`. This notebook uses the **pure-NumPy fallback** where
possible; gradient steps require the compiled C backend.

In [ ]:
import numpy as np
from SneppX_ALG import (
    Tensor, Module, Sequential, Linear, LayerNorm, GELU, Dropout,
    AdamW, CrossEntropyLoss, CosineAnnealingLR, Profiler, Timer,
)
import SneppX_ALG as S
HAS_C = S._HAS_C_BACKEND
print('C backend:', HAS_C)

## 1. Synthetic data

In [ ]:
def make_data(n=640):
    rng = np.random.default_rng(0)
    X = rng.standard_normal((n, 64)).astype(np.float32)
    y = rng.integers(0, 10, size=(n,)).astype(np.int64)
    return Tensor.from_numpy(X), Tensor.from_numpy(y)

X, y = make_data()
print('X', X.shape, 'y', y.shape)

## 2. The model

In [ ]:
class Classifier(Module):
    def __init__(self, in_dim=64, hidden=128, classes=10):
        super().__init__()
        self.net = Sequential(
            Linear(in_dim, hidden), GELU(),
            LayerNorm(hidden),
            Linear(hidden, hidden), GELU(),
            Dropout(0.1),
            Linear(hidden, classes),
        )
    def forward(self, x):
        return self.net(x)

model = Classifier()
print('params:', sum(p.numel for p in model.parameters()))

## 3. Training loop (C backend required for backward)

In [ ]:
opt = AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, min_lr=1e-5, max_lr=2e-3, total_steps=200)
prof = Profiler(enabled=True)

for step in range(200):
    idx = np.random.randint(0, X.shape[0], 64)
    xb, yb = X[idx], y[idx]
    with Timer(prof, 'forward'):
        logits = model(xb)
        loss = CrossEntropyLoss()(logits, yb)
    if not HAS_C:
        if step == 0:
            print('C backend not available - run cmake --build build --config Release')
        break
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if step % 25 == 0:
        print(step, round(loss.item(), 4))
prof.print_summary()